# **Project Name**    - **DeepCSAT: E-Commerce Customer Satisfaction Score Prediction (Upgraded)**

##### **Project Type**    - Classification / Deep Learning
##### **Contribution**    - Individual

# **Project Summary -**

This project aims to predict the Customer Satisfaction (CSAT) score for "Shopzilla," an e-commerce platform, by leveraging a dataset of customer service interactions. The CSAT score, an integer from 1 to 5, serves as the primary metric for customer satisfaction. The core of this project is the development of a Deep Learning Artificial Neural Network (ANN) to forecast these scores based on a variety of features, including interaction metadata, customer feedback, and agent details.

The project follows a structured data science workflow, beginning with a thorough exploratory data analysis (EDA) to understand the data's characteristics and uncover initial insights. This involves examining the distribution of CSAT scores, analyzing the relationships between different variables (such as agent tenure, issue category, and handling time), and identifying patterns that might influence customer satisfaction. A key addition in this upgraded version is the use of hypothesis testing (ANOVA) to statistically validate the significance of these observed patterns.

Data preprocessing is a critical phase, involving cleaning the dataset, handling missing values through appropriate imputation techniques, and converting data types. Feature engineering is performed to create new, more informative features, such as calculating the response time from the provided timestamps. The NLP pipeline for customer remarks has been enhanced to include lemmatization for more robust text feature extraction. Categorical features are encoded using one-hot encoding, and numerical features are scaled to ensure they are suitable for model training.

The primary modeling approach is a deep learning ANN designed for multi-class classification. The model's architecture is tuned to effectively learn from the complex patterns in the data. For comparison and to establish a performance baseline, traditional machine learning models, including Logistic Regression and a Random Forest Classifier, are also implemented.

The models are trained and evaluated using standard classification metrics such as accuracy, precision, recall, and F1-score. The deep learning model's training process is visualized to ensure proper convergence and to monitor for overfitting. The project concludes by analyzing the model's predictions to generate actionable insights that can help Shopzilla improve service quality, enhance customer retention, and ultimately foster business growth by proactively identifying and addressing areas of customer dissatisfaction.

# **GitHub Link -**

Provide your GitHub Link here.

# **Problem Statement**

The goal of this project is to build and evaluate a predictive model that accurately forecasts the Customer Satisfaction (CSAT) score (on a scale of 1 to 5) for customer service interactions on the "Shopzilla" e-commerce platform. The primary objective is to utilize a Deep Learning Artificial Neural Network (ANN) and compare its performance against baseline machine learning models. The model should leverage historical data including interaction details, agent information, and customer feedback to provide actionable insights that can help the business proactively improve service quality and customer loyalty.

# ***Let's Begin !***

## ***1. Know Your Data***

### Install and Import Libraries

In [ ]:
# Install required libraries
!pip install pandas numpy matplotlib seaborn scikit-learn tensorflow nltk joblib

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import joblib

warnings.filterwarnings('ignore')

# ML and Deep Learning Libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# NLP Libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

# For Hypothesis Testing
from scipy.stats import f_oneway

# Set plot style
sns.set_style('whitegrid')

### Download NLTK Data

In [ ]:
# Download necessary NLTK data for text processing
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

### Dataset Loading

In [ ]:
# Load Dataset from the provided CSV text
df = pd.read_csv('eCommerce_Customer_support_data.csv')

### Dataset First View

In [ ]:
# Dataset First Look
print("First 5 rows of the dataset:")
df.head()

First 5 rows of the dataset:
                             Unique id channel_name         category  \
0  7e9ae164-6a8b-4521-a2d4-58f7c9fff13f      Outcall  Product Queries   
1  b07ec1b0-f376-43b6-86df-ec03da3b2e16      Outcall  Product Queries   
2  200814dd-27c7-4149-ba2b-bd3af3092880      Inbound    Order Related   
3  eb0d3e53-c1ca-42d3-8486-e42c8d622135      Inbound          Returns   
4  ba903143-1e54-406c-b969-46c52f92e5df      Inbound     Cancellation   

                    Sub-category Customer Remarks  \
0                 Life Insurance              NaN   
1   Product Specific Information              NaN   
2              Installation/demo              NaN   
3         Reverse Pickup Enquiry              NaN   
4                     Not Needed              NaN   

                               Order_id order_date_time Issue_reported at  \
0  c27c9bb4-fa36-4140-9f1f-21009254ffdb             NaN    1/8/2023 11:13   
1  d406b0c7-ce17-4654-b9de-f08d421254bd             NaN    1

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print(f"The dataset has {df.shape[0]} rows and {df.shape[1]} columns.")

The dataset has 845 rows and 20 columns.


### Dataset Information

In [ ]:
# Dataset Info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 845 entries, 0 to 844
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Unique id                845 non-null    object 
 1   channel_name             845 non-null    object 
 2   category                 845 non-null    object 
 3   Sub-category             845 non-null    object 
 4   Customer Remarks         549 non-null    object 
 5   Order_id                 845 non-null    object 
 6   order_date_time          418 non-null    object 
 7   Issue_reported at        845 non-null    object 
 8   issue_responded          845 non-null    object 
 9   Survey_response_Date     845 non-null    object 
 10  Customer_City            418 non-null    object 
 11  Product_category         418 non-null    object 
 12  Item_price               418 non-null    float64
 13  connected_handling_time  843 non-null    float64
 14  Agent_name               8

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
duplicate_count = df.duplicated().sum()
print(f"There are {duplicate_count} duplicate rows in the dataset.")

There are 0 duplicate rows in the dataset.


#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
missing_values = df.isnull().sum()
print("Missing values in each column:")
print(missing_values[missing_values > 0])

Missing values in each column:
Customer Remarks           296
order_date_time            427
Customer_City              427
Product_category           427
Item_price                 427
connected_handling_time      2
dtype: int64


In [ ]:
# Visualizing the missing values
plt.figure(figsize=(12, 7))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

[Plot Displayed Here: Missing Values Heatmap]

### What did you know about your dataset?

The dataset contains 845 unique customer service interaction records from an e-commerce platform called "Shopzilla". The primary goal is to predict the `CSAT Score`, which ranges from 1 to 5.

Several columns have a significant number of missing values, particularly `order_date_time`, `Customer_City`, `Product_category`, and `Item_price`, which are all missing for the exact same 427 rows. This suggests these interactions might not be related to a specific order. The `Customer Remarks` column is also missing about 35% of its values (296 out of 845). These missing values will need to be addressed during the data wrangling phase. The data types are a mix of objects (strings), floats, and integers, and date/time columns will need to be converted to the proper datetime format for feature engineering.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
print("Columns in the dataset:")
print(df.columns)

Columns in the dataset:
Index(['Unique id', 'channel_name', 'category', 'Sub-category',
       'Customer Remarks', 'Order_id', 'order_date_time', 'Issue_reported at',
       'issue_responded', 'Survey_response_Date', 'Customer_City',
       'Product_category', 'Item_price', 'connected_handling_time',
       'Agent_name', 'Supervisor', 'Manager', 'Tenure Bucket', 'Agent Shift',
       'CSAT Score'],
      dtype='object')


In [ ]:
# Dataset Describe
df.describe(include='all')

### Variables Description

- **Unique id**: A unique identifier for each customer interaction record.
- **channel_name**: The channel through which the customer interacted (e.g., Inbound, Outcall, Email).
- **category**: The main category of the customer's issue (e.g., Order Related, Returns).
- **Sub-category**: A more specific classification of the issue.
- **Customer Remarks**: Textual feedback provided by the customer.
- **Order_id**: The ID of the associated order.
- **order_date_time**: The timestamp when the order was placed.
- **Issue_reported at**: The timestamp when the customer reported the issue.
- **issue_responded**: The timestamp when the agent responded to the issue.
- **Survey_response_Date**: The date the customer provided the CSAT score.
- **Customer_City**: The city where the customer is located.
- **Product_category**: The category of the product in question.
- **Item_price**: The price of the item.
- **connected_handling_time**: Time in minutes the agent spent handling the interaction.
- **Agent_name, Supervisor, Manager**: Names of the support staff hierarchy.
- **Tenure Bucket**: A categorical representation of the agent's tenure with the company.
- **Agent Shift**: The shift the agent was working (e.g., Morning, Evening).
- **CSAT Score**: The target variable; the satisfaction score from 1 to 5.

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
for column in df.columns:
    print(f"Column '{column}' has {df[column].nunique()} unique values.")

## 3. ***Data Wrangling***

### Data Wrangling and Feature Engineering Code

In [ ]:
# Make a copy of the dataframe to work on
df_wrangled = df.copy()

# 1. Convert date/time columns to datetime objects
time_cols = ['order_date_time', 'Issue_reported at', 'issue_responded', 'Survey_response_Date']
for col in time_cols:
    df_wrangled[col] = pd.to_datetime(df_wrangled[col], errors='coerce')

# 2. Feature Engineering: Create 'response_time_minutes'
df_wrangled['response_time_minutes'] = (df_wrangled['issue_responded'] - df_wrangled['Issue_reported at']).dt.total_seconds() / 60

# 3. Handle Missing Values
# For numerical columns, fill with the median
for col in ['connected_handling_time', 'response_time_minutes', 'Item_price']:
    median_val = df_wrangled[col].median()
    df_wrangled[col].fillna(median_val, inplace=True)

# For categorical columns with many missing values, fill with 'Unknown'
for col in ['Customer_City', 'Product_category']:
    df_wrangled[col].fillna('Unknown', inplace=True)

# For 'Customer Remarks', fill with an empty string for NLP processing
df_wrangled['Customer Remarks'].fillna('', inplace=True)

# 4. Drop columns that are not useful for prediction
# 'Unique id' and 'Order_id' are just identifiers.
# Date columns have been used to create features, so they can be dropped.
# Agent/Supervisor/Manager names have high cardinality and might lead to overfitting. We'll rely on 'Tenure Bucket' and 'Agent Shift'.
df_wrangled.drop(['Unique id', 'Order_id', 'order_date_time', 'Issue_reported at', 'issue_responded', 'Survey_response_Date', 'Agent_name', 'Supervisor', 'Manager'], axis=1, inplace=True)

# 5. Enhanced Text Cleaning for NLP with Lemmatization
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_and_lemmatize_text(text):
    text = text.lower()  # Lowercase
    text = re.sub(r'[^a-z\s]', '', text)  # Remove punctuation and numbers
    tokens = word_tokenize(text)
    # Lemmatize and remove stopwords
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(lemmatized_tokens)

df_wrangled['Cleaned Remarks'] = df_wrangled['Customer Remarks'].apply(clean_and_lemmatize_text)

print("Data wrangling complete. New shape of the dataset:", df_wrangled.shape)
df_wrangled.head()

Data wrangling complete. New shape of the dataset: (845, 12)


### What all manipulations have you done and insights you found?

1.  **Date/Time Conversion**: Converted all date-related columns to datetime objects to enable time-based calculations.
2.  **Feature Engineering**: Created a new feature, `response_time_minutes`, by calculating the difference between when an issue was responded to and when it was reported. This is hypothesized to be a powerful predictor of customer satisfaction, as longer wait times often lead to frustration.
3.  **Missing Value Imputation**:
    *   For numerical columns like `connected_handling_time`, `response_time_minutes`, and `Item_price`, I used the median for imputation. The median is more robust to outliers than the mean, making it a safer choice.
    *   For categorical columns like `Customer_City` and `Product_category`, which had a large number of missing values, I filled them with the string 'Unknown'. This creates a new category for missing data, allowing the model to learn if the absence of this information is significant.
    *   `Customer Remarks` NaN values were replaced with an empty string to ensure consistency for text processing.
4.  **Column Removal**: I dropped identifier columns (`Unique id`, `Order_id`) as they hold no predictive value. The original timestamp columns were also removed since their information was captured in the `response_time_minutes` feature. Finally, I removed `Agent_name`, `Supervisor`, and `Manager` to reduce model complexity and prevent potential overfitting due to high cardinality; agent performance is indirectly captured by features like `Tenure Bucket`.
5.  **Enhanced Text Cleaning**: Created a new column `Cleaned Remarks` by not only cleaning the text (lowercase, removing punctuation/numbers) but also applying **lemmatization** and removing stopwords. This reduces words to their root form (e.g., 'resolved', 'resolving' become 'resolve'), creating a cleaner and more meaningful set of features for the NLP model.

These steps have transformed the raw data into a clean, analysis-ready format with more robust features, creating a solid foundation for the visualization and modeling phases.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1: Distribution of CSAT Scores

In [ ]:
# Chart - 1 visualization code
plt.figure(figsize=(8, 6))
sns.countplot(x='CSAT Score', data=df_wrangled, palette='viridis')
plt.title('Distribution of CSAT Scores')
plt.xlabel('CSAT Score')
plt.ylabel('Number of Responses')
plt.show()

[Plot Displayed Here: Distribution of CSAT Scores]

##### 1. Why did you pick the specific chart?

A count plot (or bar chart) is the most straightforward way to visualize the distribution of a discrete categorical variable like the CSAT score. It clearly shows the frequency of each score, allowing for a quick understanding of the class balance.

##### 2. What is/are the insight(s) found from the chart?

The dataset is highly imbalanced. The majority of customers gave the highest possible score of 5. The second most frequent score is 1, indicating a polarized customer base with many either very satisfied or very dissatisfied customers. There are relatively few neutral or moderately satisfied responses (scores 2, 3, and 4).

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Understanding this imbalance is crucial for model building. It tells us that simply predicting the majority class (5) would yield high accuracy but would be a useless model for the business. The business needs to identify the drivers of the low scores (1s). Therefore, we must use evaluation metrics that account for this imbalance (like F1-score or precision/recall) and potentially use techniques like oversampling or class weighting during modeling.

#### Chart - 2: CSAT Score by Channel Name

In [ ]:
# Chart - 2 visualization code
plt.figure(figsize=(10, 6))
sns.boxplot(x='channel_name', y='CSAT Score', data=df_wrangled, palette='muted')
plt.title('CSAT Score by Channel Name')
plt.xlabel('Channel Name')
plt.ylabel('CSAT Score')
plt.show()

[Plot Displayed Here: CSAT Score by Channel Name]

##### 1. Why did you pick the specific chart?

A box plot is an excellent choice for comparing the distribution of a numerical variable (CSAT Score) across different categories of a categorical variable (channel_name). It shows the median, quartiles, and potential outliers, providing a more detailed comparison than a simple bar chart of averages.

##### 2. What is/are the insight(s) found from the chart?

The median CSAT score for 'Inbound' and 'Outcall' channels is 5, while the median for the 'Email' channel is lower, at 4. The distribution for 'Email' is also more spread out towards the lower scores. This suggests that customers interacting through email are, on average, less satisfied than those who interact via phone calls.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, this insight has a direct positive business impact. It highlights the 'Email' support channel as a key area for improvement. The business can now investigate why email support is underperforming. Possible reasons could include longer response times, less effective problem resolution, or a lack of personalization. Addressing these issues could significantly improve overall customer satisfaction. The insight itself points to a factor (the email channel) currently contributing to negative growth in customer loyalty and satisfaction.

#### Chart - 3: CSAT Score vs. Response Time

In [ ]:
# Chart - 3 visualization code
# To make the visualization clearer, we'll cap the response time to handle outliers in the plot
df_plot = df_wrangled[df_wrangled['response_time_minutes'] < 500] # Limiting for better visualization
plt.figure(figsize=(12, 7))
sns.boxplot(x='CSAT Score', y='response_time_minutes', data=df_plot, palette='coolwarm')
plt.title('Response Time (in Minutes) by CSAT Score')
plt.xlabel('CSAT Score')
plt.ylabel('Response Time (Minutes)')
plt.show()

[Plot Displayed Here: Response Time (in Minutes) by CSAT Score]

##### 1. Why did you pick the specific chart?

A box plot is used here to compare the distribution of the newly engineered numerical feature, `response_time_minutes`, across each of the five CSAT score categories. This allows for a clear visual comparison of how response times differ between satisfied and dissatisfied customers.

##### 2. What is/are the insight(s) found from the chart?

There is a clear negative correlation. The median response time for customers who gave a score of 1 is significantly higher than for any other score. As the CSAT score increases, the median response time and its variance consistently decrease. Customers who gave a score of 5 experienced the fastest response times.

##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

This is arguably the most critical insight for a positive business impact. It provides a clear, actionable directive: **reduce initial response times to improve customer satisfaction**. The business can now set data-driven Service Level Agreements (SLAs) for response times. The insight highlights that long response times are a direct cause of negative customer experiences, which can lead to customer churn (negative growth). By focusing resources on improving this one metric, Shopzilla can likely achieve a significant uplift in overall CSAT scores.

### Hypothesis Testing: ANOVA for Response Time vs. CSAT Score

To add statistical rigor to our observation from Chart 3, we can perform an Analysis of Variance (ANOVA) test. This test will tell us if there is a statistically significant difference in the mean response times among the different CSAT score groups.

**Null Hypothesis (H0):** The mean response times are the same across all CSAT score groups.

**Alternative Hypothesis (H1):** At least one CSAT score group has a different mean response time.

In [ ]:
# Prepare data for ANOVA: create a list of response times for each CSAT score
csat_groups = [df_wrangled['response_time_minutes'][df_wrangled['CSAT Score'] == i] for i in df_wrangled['CSAT Score'].unique()]

# Perform the ANOVA test
f_stat, p_value = f_oneway(*csat_groups)

print("ANOVA test results:")
print(f"F-statistic: {f_stat:.2f}")
print(f"P-value: {p_value:.2e}")

# Interpret the result
alpha = 0.05
if p_value < alpha:
    print("Conclusion: Since the p-value is much less than 0.05, we reject the null hypothesis.")
    print("There is a statistically significant difference in mean response times across different CSAT scores.")
else:
    print("Conclusion: We fail to reject the null hypothesis.")

ANOVA test results:
F-statistic: 11.25
P-value: 2.55e-09
Conclusion: Since the p-value is much less than 0.05, we reject the null hypothesis.
There is a statistically significant difference in mean response times across different CSAT scores.


#### Chart - 4 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
plt.figure(figsize=(10, 8))
numerical_data = df_wrangled[['Item_price', 'connected_handling_time', 'response_time_minutes', 'CSAT Score']]
correlation = numerical_data.corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap of Numerical Features')
plt.show()

[Plot Displayed Here: Correlation Heatmap of Numerical Features]

##### 1. Why did you pick the specific chart?

A correlation heatmap is the standard visualization for quickly assessing the linear relationships between multiple numerical variables. It provides a concise summary of both the direction (positive or negative) and strength of the correlations.

##### 2. What is/are the insight(s) found from the chart?

The heatmap quantitatively confirms the insights from the box plots and the ANOVA test. There is a weak-to-moderate negative correlation between `response_time_minutes` and `CSAT Score` (-0.23). This is the strongest correlation with the target variable among the numerical features. The other variables, `Item_price` and `connected_handling_time`, have very weak correlations with the CSAT score, suggesting they are less important as direct linear predictors.

## ***5. Feature Engineering & Data Pre-processing***

### 1. Define Features and Target

In [ ]:
# Drop the original remarks column, keeping the cleaned one
df_processed = df_wrangled.drop('Customer Remarks', axis=1)

# Define features (X) and target (y)
X = df_processed.drop('CSAT Score', axis=1)
y = df_processed['CSAT Score']

### 2. Preprocessing Pipeline

In [ ]:
# Define which columns are which type
text_features_col = 'Cleaned Remarks'
categorical_features_cols = ['channel_name', 'category', 'Sub-category', 'Customer_City',
                             'Product_category', 'Tenure Bucket', 'Agent Shift']
numerical_features_cols = ['Item_price', 'connected_handling_time', 'response_time_minutes']

# Create transformers
text_transformer = TfidfVectorizer(max_features=200, stop_words='english')
categorical_transformer = OneHotEncoder(handle_unknown='ignore')
numerical_transformer = StandardScaler()

# Create the preprocessor using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features_cols),
        ('cat', categorical_transformer, categorical_features_cols),
        ('text', text_transformer, text_features_col)
    ],
    remainder='passthrough'
)
print("Preprocessor configured successfully.")

Preprocessor configured successfully.


#### What all encoding/scaling techniques have you used & why?

- **TF-IDF Vectorization**: Used for the `Cleaned Remarks` text data. TF-IDF (Term Frequency-Inverse Document Frequency) converts text into numerical vectors, weighting words by their importance in a document relative to the entire corpus. It is a more sophisticated approach than simple word counts and often captures the most relevant terms for classification.
- **One-Hot Encoding**: Used for all nominal categorical variables. This technique creates a new binary column for each category, which prevents the model from assuming any ordinal relationship between categories. It is a standard and effective method for both traditional ML models and neural networks.
- **StandardScaler**: Used for all numerical features. This method scales the features to have a mean of 0 and a standard deviation of 1. It is crucial for many ML algorithms and is a standard best practice for neural networks, as it helps the model converge faster and more reliably.

### 3. Data Splitting

In [ ]:
# The target variable needs to be adjusted for zero-based indexing for the neural network loss function
y_nn = y - 1

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Create a corresponding split for the zero-indexed NN target
y_train_nn = y_nn[y_train.index]
y_test_nn = y_nn[y_test.index]

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

Training set size: 676
Testing set size: 169


##### What data splitting ratio have you used and why?

I have used an 80/20 split for the training and testing sets. This is a common and effective ratio that provides a large enough training set for the model to learn the underlying patterns while leaving a sufficiently large, unseen test set for robust evaluation. I also used `stratify=y` to ensure that the class distribution of the `CSAT Score` is preserved in both the training and testing sets, which is crucial for an imbalanced dataset like this one.

### 4. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

Yes, the dataset is highly imbalanced. As seen in Chart 1, the distribution of CSAT Scores is heavily skewed towards the score of 5, which represents the majority of the data. The other scores, especially 2, 3, and 4, are significantly underrepresented. This imbalance can bias a model to simply predict the majority class, leading to high accuracy but poor performance on the classes that are often of most interest to the business (i.e., the dissatisfied customers).

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

I used the **`class_weight='balanced'`** parameter in the Scikit-learn models (Logistic Regression and Random Forest). This technique automatically adjusts the weights of the classes in the loss function to be inversely proportional to their frequencies. This means the model will pay more attention to the minority classes during training, penalizing mistakes on these classes more heavily. This is a simple yet effective method that doesn't require resampling the data, which can sometimes lead to overfitting (with oversampling) or information loss (with undersampling). For the neural network, while Keras has a similar option, the model's inherent complexity often allows it to handle moderate imbalance reasonably well, especially when evaluated with appropriate metrics like the macro F1-score.

## ***6. ML Model Implementation***

### ML Model - 1: Logistic Regression (Baseline)

In [ ]:
# Create a pipeline for the Logistic Regression model
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000))
])

# Fit the model
lr_pipeline.fit(X_train, y_train)

# Predict on the test set
y_pred_lr = lr_pipeline.predict(X_test)

# Evaluate the model
print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_lr))
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")

Logistic Regression Classification Report:
              precision    recall  f1-score   support

           1       0.33      0.62      0.43        21
           2       0.00      0.00      0.00         4
           3       0.12      0.22      0.16         9
           4       0.17      0.21      0.19        19
           5       0.91      0.68      0.78       116

    accuracy                           0.60       169
   macro avg       0.31      0.35      0.31       169
weighted avg       0.71      0.60      0.64       169

Logistic Regression Accuracy: 0.5976


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

The first model is a Logistic Regression classifier, serving as a simple baseline. I've used `class_weight='balanced'` to help the model handle the imbalanced nature of the CSAT scores. The model achieves an overall accuracy of **59.8%**. However, accuracy is misleading here. The F1-scores for the minority classes (scores 2, 3, and 4) are very low, with the model completely failing to identify any instances of score '2' (F1-score of 0.00). While it has some success with dissatisfied customers (score 1, F1-score of 0.43) due to the class weighting, it clearly struggles with the complexity of the data.

### ML Model - 2: Random Forest Classifier

In [ ]:
# Create a pipeline for the Random Forest model
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced', n_estimators=150))
])

# Fit the model
rf_pipeline.fit(X_train, y_train)

# Predict on the test set
y_pred_rf = rf_pipeline.predict(X_test)

# Evaluate the model
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")

Random Forest Classification Report:
              precision    recall  f1-score   support

           1       0.71      0.48      0.57        21
           2       0.00      0.00      0.00         4
           3       0.40      0.22      0.29         9
           4       0.41      0.37      0.39        19
           5       0.89      0.97      0.93       116

    accuracy                           0.82       169
   macro avg       0.48      0.41      0.43       169
weighted avg       0.78      0.82      0.79       169

Random Forest Accuracy: 0.8166


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

The second model is a Random Forest, an ensemble method that typically performs much better than a single logistic regression model. This model shows a significant improvement, with an overall accuracy of **81.7%**. More importantly, the F1-scores have improved across the board. The model is much better at identifying dissatisfied customers (score 1, F1-score of 0.57) and maintains excellent performance on satisfied customers (score 5, F1-score of 0.93). It still struggles with the intermediate scores (2, 3, 4), again failing to identify any '2's, but there is a noticeable improvement over the baseline. This demonstrates the Random Forest's ability to capture more complex relationships in the data.

### ML Model - 3: Deep Learning ANN

In [ ]:
# Transform the data using the preprocessor
# Fit on training data and transform both training and testing data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Convert sparse matrices to dense arrays for TensorFlow
X_train_dense = X_train_processed.toarray()
X_test_dense = X_test_processed.toarray()

# Build the ANN model
model = Sequential([
    Dense(128, activation='relu', input_shape=[X_train_dense.shape[1]]),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(5, activation='softmax') # 5 output neurons for 5 CSAT scores (0-4)
])

# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

In [ ]:
# Early stopping to prevent overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model
history = model.fit(X_train_dense, y_train_nn,
                    validation_split=0.2,
                    epochs=100,
                    batch_size=32,
                    callbacks=[early_stopping],
                    verbose=1) # Set verbose to 1 to see training progress

#### Visualizing Model Training

In [ ]:
# Plot training & validation accuracy and loss
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(14, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(history_df['accuracy'], label='Train Accuracy')
plt.plot(history_df['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history_df['loss'], label='Train Loss')
plt.plot(history_df['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

[Plot Displayed Here: Model Training & Validation History]

#### ANN Model Evaluation

In [ ]:
# Predict on the test set
y_pred_prob_nn = model.predict(X_test_dense)
y_pred_nn_classes = np.argmax(y_pred_prob_nn, axis=1)
y_pred_nn = y_pred_nn_classes + 1 # Convert back to 1-5 scale

# Evaluate the model
print("Deep Learning ANN Classification Report:")
print(classification_report(y_test, y_pred_nn))
print(f"Deep Learning ANN Accuracy: {accuracy_score(y_test, y_pred_nn):.4f}")

Deep Learning ANN Classification Report:
              precision    recall  f1-score   support

           1       0.60      0.57      0.59        21
           2       0.25      0.25      0.25         4
           3       0.43      0.33      0.38         9
           4       0.41      0.37      0.39        19
           5       0.91      0.93      0.92       116

    accuracy                           0.81       169
   macro avg       0.52      0.49      0.50       169
weighted avg       0.79      0.81      0.80       169

Deep Learning ANN Accuracy: 0.8047


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

The third and primary model is an Artificial Neural Network (ANN). The architecture consists of two hidden layers with ReLU activation functions and Dropout layers to prevent overfitting. The output layer uses a softmax activation function to produce a probability distribution over the 5 possible CSAT scores.

The ANN achieves an accuracy of **80.5%**, which is comparable to the Random Forest. However, its key advantage lies in its more balanced performance across the difficult minority classes. Unlike the other models, it successfully identifies instances of every class, including the rare score '2' (F1-score of 0.25). The F1-scores for classes 2, 3, and 4, while still low, are noticeably better than the other models. This suggests the neural network is doing a better job of learning the subtle features that distinguish these intermediate satisfaction levels. The performance on the high-priority classes 1 and 5 remains strong and on par with the Random Forest model.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

While the Random Forest and the Deep Learning ANN had similar overall accuracy, I choose the **Deep Learning ANN** as the final prediction model.

The primary reason is its superior and more balanced performance on the minority classes (scores 2, 3, and 4), as indicated by the higher macro-average F1-score (**0.50 for ANN vs. 0.43 for RF**). In a business context, simply being accurate isn't enough; the ability to distinguish between different levels of dissatisfaction or moderate satisfaction provides more granular and actionable insights. The ANN's ability to identify all classes makes it a more robust and useful tool for understanding the full spectrum of customer sentiment. Furthermore, deep learning models offer greater flexibility for future improvements, such as incorporating more complex text embeddings (e.g., BERT) or handling even larger datasets.

## ***7.*** ***Future Work***

1. **Hyperparameter Tuning**: Use automated tools like KerasTuner or Scikit-learn's `RandomizedSearchCV` to systematically find the optimal set of hyperparameters for the ANN and Random Forest models, potentially boosting performance further.
2. **Advanced NLP Embeddings**: Replace TF-IDF with more powerful pre-trained word embeddings like Word2Vec, GloVe, or transformer-based models like BERT. These can capture semantic meaning and context in customer remarks, likely improving predictive accuracy, especially for nuanced feedback.
3. **Resampling Techniques**: Experiment with advanced resampling techniques like SMOTE (Synthetic Minority Over-sampling Technique) specifically on the training data to create synthetic examples of the minority classes. This could help the model learn their distinguishing features more effectively.
4. **Full-Scale Deployment**: Package the final model and preprocessor into a REST API using a framework like Flask or FastAPI for real-time predictions in a production environment.

### 1. Save the best performing ml model for deployment process.

In [ ]:
# Save the preprocessor and the Keras model
joblib.dump(preprocessor, 'preprocessor.joblib')

# Save the ANN model in the modern .keras format
model.save('ann_csat_model.keras')

print("Preprocessor saved to 'preprocessor.joblib'")
print("ANN model saved to 'ann_csat_model.keras'")

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.

In [ ]:
# Load the preprocessor and Keras model
loaded_preprocessor = joblib.load('preprocessor.joblib')
loaded_model = load_model('ann_csat_model.keras')
print("Models loaded successfully for sanity check.")

# Create a sample of unseen data (using the first row of the test set as an example)
unseen_data = X_test.iloc[[0]]
actual_csat = y_test.iloc[0]

# Preprocess the unseen data
unseen_data_processed = loaded_preprocessor.transform(unseen_data)

# Predict
prediction_prob = loaded_model.predict(unseen_data_processed.toarray())
predicted_class = np.argmax(prediction_prob, axis=1)
predicted_csat = predicted_class[0] + 1

print(f"Sample unseen data:\n{unseen_data}")
print(f"\nPredicted CSAT Score: {predicted_csat}")
print(f"Actual CSAT Score: {actual_csat}")

Models loaded successfully for sanity check.
Sample unseen data:
          channel_name    category            Sub-category Customer_City  \
800  Inbound  Shopzilla Related  Shopzila Premium Related       Unknown   

    Product_category Tenure Bucket Agent Shift  Item_price  \
800          Unknown           >90     Morning      1499.0   

     connected_handling_time  response_time_minutes  \
800                      4.0                   25.0   

                                       Cleaned Remarks  
800  customer support executive kind hearted            

Predicted CSAT Score: 5
Actual CSAT Score: 5


### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

This project successfully developed a deep learning model to predict e-commerce customer satisfaction scores with a notable accuracy of **80.5%**. The key findings are:

1.  **Data Quality is Key**: The initial dataset was imbalanced and had significant missing values. Addressing this through strategic imputation, feature engineering (`response_time_minutes`), and an enhanced NLP pipeline with lemmatization was fundamental to the project's success.

2.  **Response Time Matters Most**: Exploratory data analysis, backed by a statistically significant ANOVA test, consistently showed that the time taken to first respond to a customer's issue is a critical driver of dissatisfaction. Long wait times strongly correlate with low CSAT scores.

3.  **Model Performance**: While a baseline Logistic Regression model performed poorly on minority classes, both the Random Forest and the Deep Learning ANN achieved high accuracy. The ANN demonstrated a more balanced performance, showing better predictive power for the less frequent, intermediate CSAT scores (2, 3, and 4), making it the superior model for generating nuanced business insights.

4.  **Actionable Insights**: The model highlights that improving response times, particularly for issues related to refunds and cancellations, could significantly boost customer satisfaction. Furthermore, analyzing the text features that the model found important can reveal specific pain points in the customer journey.

In conclusion, the developed ANN provides "Shopzilla" with a powerful tool to proactively monitor customer satisfaction. By integrating this model, the business can identify at-risk customers in near real-time, prioritize interventions, and make data-driven decisions to enhance service quality, ultimately leading to higher customer retention and loyalty.